# FLUX.2 klein + ComfyUI on Colab
Runtime > Change runtime type > **GPU** first. Then run the cells top to bottom. After a disconnect: reconnect and run them again (models are re-used if you chose Drive).

In [ ]:
# 1) GPU check (should list an NVIDIA GPU)
!nvidia-smi

In [ ]:
# 2) Settings + optional secrets/Drive.  PERSIST = "none" (temporary disk) or "drive" (keep models on Google Drive)
import os
GITHUB_USER = "aiexpert807567-dotcom"      # <- set by bootstrap.sh
REPO = "flux-comfyui-colab"
PERSIST = "none"                      # "drive" to keep models/outputs across sessions
TUNNEL = "cloudflared"                # "none" = no public URL (use the Generate cell below)

os.environ["PERSIST"] = PERSIST
os.environ["TUNNEL"] = TUNNEL
try:  # optional: Colab Secrets (key icon, left sidebar). Add HF_TOKEN / GITHUB_TOKEN there, never in code.
    from google.colab import userdata
    for k in ("HF_TOKEN", "GITHUB_TOKEN"):
        try: os.environ[k] = userdata.get(k)
        except Exception: pass
except ImportError:
    pass
if PERSIST == "drive":
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# 3) Clone (or update) the repo
import os
tok = os.environ.get("GITHUB_TOKEN")
url = f"https://{tok+'@' if tok else ''}github.com/{GITHUB_USER}/{REPO}.git"
if os.path.isdir(f"/content/{REPO}/.git"):
    !git -C /content/{REPO} pull --ff-only
else:
    !git clone {url} /content/{REPO}
%cd /content/{REPO}

In [ ]:
# 4) ONE-COMMAND SETUP: detect GPU, install ComfyUI, download models, health check, start server (+ tunnel URL)
!bash colab/setup_colab.sh

Open the **ComfyUI URL** printed above (anyone with that link controls your session - don't share it).

In [ ]:
# 5) Headless test: text -> image, shown inline (works even with TUNNEL="none")
!python scripts/run_workflow.py workflows/flux_basic.json --prompt "a lighthouse on a cliff at sunrise, dramatic clouds, photograph" --seed 7
from IPython.display import Image, display
import glob
f = sorted(glob.glob("outputs/*.png"))[-1]; display(Image(f))

In [ ]:
# 6) Image-to-image: upload a file with the folder icon on the left, then set the path
IMG = "/content/my_photo.png"
!python scripts/run_workflow.py workflows/flux_image_to_image.json --image {IMG} --prompt "watercolor painting" --denoise 0.55 --seed 1

In [ ]:
# 7) Optional upscaler (adds ~70 MB)
!WITH_UPSCALER=1 bash colab/install_models.sh
!python scripts/run_workflow.py workflows/flux_upscale.json --image {IMG}